# Notebook 3: Data Cleaning and Saving the Final File
**Input:** `hospital_data.csv`  →  **Output:** `hospital_data_cleaned.csv`

Steps: inspect → fix text → remove duplicates → fix invalid values → handle missing values → feature engineering → validate → save → download.

In [51]:
try:
    from google.colab import files
    files.upload()   # choose hospital_data.csv
except ImportError:
    print("Not running in Colab - skipping upload.")

Not running in Colab - skipping upload.


In [52]:
import pandas as pd
import numpy as np

df = pd.read_csv("hospital_data.csv")
raw = df.copy()          # keep a copy to compare before/after
print("Original shape:", df.shape)
df.head()

Original shape: (255, 11)


,Patient_ID,Age,Gender,City,Department,Admission_Type,Num_Comorbidities,Systolic_BP,Waiting_Time_Mins,Admission_Date,Length_of_Stay_Days
0,P1050,68.0,Male,Ghaziabad,Surgery,Planned,2,134.0,40.0,2025-05-09,7
1,P1147,33.0,Male,Pune,Orthopedics,Planned,2,117.0,39.0,2025-01-24,6
2,P1205,61.0,Female,Pune,Orthopedics,Planned,1,117.0,75.0,2025-10-17,8
3,P1187,48.0,Female,Pune,Surgery,Planned,0,116.0,15.0,2026-06-13,2
4,P1190,42.0,Male,Delhi,General Medicine,Referral,2,135.0,58.0,2026-11-08,6


## Step 1: Inspect problems

In [53]:
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nCity values:", df["City"].unique())
print("\nInvalid ages:")
print(df[(df["Age"] < 0) | (df["Age"] > 110)][["Patient_ID", "Age"]])

Missing values:
Patient_ID             0
Age                    5
Gender                 0
City                   0
Department             0
Admission_Type         5
Num_Comorbidities      0
Systolic_BP            6
Waiting_Time_Mins      4
Admission_Date         0
Length_of_Stay_Days    0
dtype: int64

Duplicate rows: 5

City values: <StringArray>
['Ghaziabad',      'Pune',     'Delhi',   'Lucknow',    'mumbai',    'Mumbai',
    ' DELHI', 'Lucknow  ', 'GHAZIABAD',    ' Delhi', 'ghaziabad',   'LUCKNOW']
Length: 12, dtype: str

Invalid ages:
    Patient_ID    Age
12       P1218  -34.0
115      P1093  214.0


## Step 2: Clean text columns (spaces and capitalisation)

In [54]:
text_cols = ["Patient_ID", "Gender", "City", "Department", "Admission_Type"]
for col in text_cols:
    df[col] = df[col].str.strip()

# 'mumbai', ' DELHI', 'GHAZIABAD' -> 'Mumbai', 'Delhi', 'Ghaziabad'
df["City"] = df["City"].str.title()

print(df["City"].value_counts())

City
Delhi        73
Lucknow      54
Mumbai       52
Ghaziabad    46
Pune         30
Name: count, dtype: int64


## Step 3: Remove duplicate rows

In [55]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows. Rows now: {len(df)}")

Removed 5 duplicate rows. Rows now: 250


## Step 4: Fix invalid values

In [56]:
# Ages must be realistic: 0 to 110. Anything else becomes missing (NaN) and is filled in Step 5.
invalid_age = (df["Age"] < 0) | (df["Age"] > 110)
print("Invalid ages found:", invalid_age.sum())
df.loc[invalid_age, "Age"] = np.nan

Invalid ages found: 2


## Step 5: Handle missing values

In [57]:
print("Missing before:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Numeric columns -> fill with median (robust to outliers)
for col in ["Age", "Systolic_BP", "Waiting_Time_Mins"]:
    df[col] = df[col].fillna(df[col].median())

# Categorical column -> label as 'Unknown'
df["Admission_Type"] = df["Admission_Type"].fillna("Unknown")

print("\nMissing after:", df.isnull().sum().sum())

Missing before:
Age                  7
Admission_Type       5
Systolic_BP          6
Waiting_Time_Mins    4
dtype: int64

Missing after: 0


## Step 6: Fix correct data types and treat outliers

In [46]:
df["Admission_Date"] = pd.to_datetime(df["Admission_Date"])
df["Age"] = df["Age"].astype(int)

# Cap extreme waiting times using the IQR rule
q1, q3 = df["Waiting_Time_Mins"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print("Waiting time upper limit:", upper)
print("Values capped:", (df["Waiting_Time_Mins"] > upper).sum())
df["Waiting_Time_Mins"] = df["Waiting_Time_Mins"].clip(upper=upper).round().astype(int)
df["Systolic_BP"] = df["Systolic_BP"].round().astype(int)

Waiting time upper limit: 161.5
Values capped: 3


## Step 7: Add useful new columns

In [47]:
df["Age_Group"] = pd.cut(df["Age"], bins=[-1, 17, 40, 60, 110],
                         labels=["Child", "Young Adult", "Middle Aged", "Senior"])
df["Stay_Category"] = np.where(df["Length_of_Stay_Days"] > 7, "Long", "Short")
df["Admission_Year"] = df["Admission_Date"].dt.year
df["Admission_Month"] = df["Admission_Date"].dt.month_name()
df.head()

,Patient_ID,Age,Gender,City,Department,Admission_Type,Num_Comorbidities,Systolic_BP,Waiting_Time_Mins,Admission_Date,Length_of_Stay_Days,Age_Group,Stay_Category,Admission_Year,Admission_Month
0,P1050,68,Male,Ghaziabad,Surgery,Planned,2,134,40,2025-05-09,7,Senior,Short,2025,May
1,P1147,33,Male,Pune,Orthopedics,Planned,2,117,39,2025-01-24,6,Young Adult,Short,2025,January
2,P1205,61,Female,Pune,Orthopedics,Planned,1,117,75,2025-10-17,8,Senior,Long,2025,October
3,P1187,48,Female,Pune,Surgery,Planned,0,116,15,2026-06-13,2,Middle Aged,Short,2026,June
4,P1190,42,Male,Delhi,General Medicine,Referral,2,135,58,2026-11-08,6,Middle Aged,Short,2026,November


## Step 8: Validate the cleaned data

In [48]:
print("Shape before:", raw.shape, "-> after:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("Age range:", df["Age"].min(), "to", df["Age"].max())
print("Cities:", sorted(df["City"].unique()))
print()
df.info()

Shape before: (255, 11) -> after: (250, 15)
Missing values: 0
Duplicates: 0
Age range: 1 to 92
Cities: ['Delhi', 'Ghaziabad', 'Lucknow', 'Mumbai', 'Pune']

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Patient_ID           250 non-null    str           
 1   Age                  250 non-null    int64         
 2   Gender               250 non-null    str           
 3   City                 250 non-null    str           
 4   Department           250 non-null    str           
 5   Admission_Type       250 non-null    str           
 6   Num_Comorbidities    250 non-null    int64         
 7   Systolic_BP          250 non-null    int64         
 8   Waiting_Time_Mins    250 non-null    int64         
 9   Admission_Date       250 non-null    datetime64[us]
 10  Length_of_Stay_Days  250 non-null    int64         
 11  Age_

## Step 9: Save the cleaned file

In [49]:
df.to_csv("hospital_data_cleaned.csv", index=False)
print("Saved: hospital_data_cleaned.csv")

# Confirm by reading it back
check = pd.read_csv("hospital_data_cleaned.csv")
print(check.shape)
check.head()

Saved: hospital_data_cleaned.csv
(250, 15)


,Patient_ID,Age,Gender,City,Department,Admission_Type,Num_Comorbidities,Systolic_BP,Waiting_Time_Mins,Admission_Date,Length_of_Stay_Days,Age_Group,Stay_Category,Admission_Year,Admission_Month
0,P1050,68,Male,Ghaziabad,Surgery,Planned,2,134,40,2025-05-09,7,Senior,Short,2025,May
1,P1147,33,Male,Pune,Orthopedics,Planned,2,117,39,2025-01-24,6,Young Adult,Short,2025,January
2,P1205,61,Female,Pune,Orthopedics,Planned,1,117,75,2025-10-17,8,Senior,Long,2025,October
3,P1187,48,Female,Pune,Surgery,Planned,0,116,15,2026-06-13,2,Middle Aged,Short,2026,June
4,P1190,42,Male,Delhi,General Medicine,Referral,2,135,58,2026-11-08,6,Middle Aged,Short,2026,November


## Step 10: How to get the cleaned file

**Google Colab:** run the cell below, and the file downloads to your computer. (You can also open the folder icon on the left sidebar, right-click `hospital_data_cleaned.csv` and choose *Download*.)

**Jupyter Notebook / JupyterLab / VS Code:** the file is saved in the same folder as this notebook. In Jupyter's file browser, tick the file and click *Download*.

In [50]:
try:
    from google.colab import files
    files.download("hospital_data_cleaned.csv")
except ImportError:
    import os
    print("Saved locally at:", os.path.abspath("hospital_data_cleaned.csv"))

Saved locally at: c:\Users\samiksha yadav\OneDrive\Desktop\DATA_ANALYTICS_PROJECT\hospital_data_cleaned.csv
